# 第04章 矩阵的解剖 · C2组人工复核

G20 / C2 · C组首轮审查 · 2026-11-16

| 被审版本 | 固定提交 SHA | 核对文件与范围 |
| --- | --- | --- |
| A1（G14） | `882b66f7afcfb14a13ae5a06500d38487e408cd2` | `A1-G14-正文.md`、`A1组-实验.ipynb`，重点核对 4.1–4.4 主张、相关单元及保存输出 |
| A2（G18） | `882b66f7afcfb14a13ae5a06500d38487e408cd2` | `A2-G18-正文.md`、`A2组-实验.ipynb`，重点核对 4.1–4.4 主张、相关单元及保存输出 |
| B1（G3） | `882b66f7afcfb14a13ae5a06500d38487e408cd2` | `B1-G3-验证报告.md`、`B1-G3-验证.ipynb`、修正清单和公开AI记录，查阅与本次算例相关的代码和保存输出 |
| B2（G13） | `882b66f7afcfb14a13ae5a06500d38487e408cd2` | `B2组-G13-修改建议.md`、`B2组-G13-正文修改版本.md`。独立验证 Notebook 的代码和输出待取得 |
| A1 补查（G14） | `7f2ba646b36eb964d54b7e9e3d65d2c7163097fe` | [PR #39](https://github.com/prince19992516/-2026/pull/39) 的正文与实验，按相关单元比对 |

固定文件链接和问题处理建议见[批判报告](C2组-批判报告.md)。所有参考值均由[独立推导](C2组-手动推导.md)中的显式矩阵、有理数或解析式预先给定。[AI陷阱记录](C2组-AI陷阱.md)对应 4.4 的两项 PCA 反例及其他教学算例。

本 Notebook 使用独立算例，已保存新内核的实测输出。数据为固定合成问题，使用 CPU、float64。复现需要 Python 3 与 NumPy，在 Jupyter 中从新内核运行全部单元。计时证据另见报告。

In [1]:
import platform
import sys

import numpy as np

ATOL = 1e-12  # 二到八维 float64 运算，阈值覆盖正常舍入，远小于所比较差异
passed = []


def check(label, condition):
    if not bool(condition):
        raise AssertionError(label)
    passed.append(label)
    print(f"通过 | {label}")


def close(label, actual, expected, atol=ATOL, rtol=0.0):
    ok = np.shape(actual) == np.shape(expected) and np.allclose(
        actual, expected, atol=atol, rtol=rtol
    )
    check(label, ok)
    print(f"       实测 {actual} | 参考 {expected}")


print(platform.platform())
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | eps={np.finfo(float).eps:.3e}")

Windows-11-10.0.22631-SP0
Python 3.12.14 | NumPy 2.3.5 | eps=2.220e-16


## 4.1 直接法 · LU 与 Cholesky

| 被审结论 | 独立预期 | 判断标准 |
| --- | --- | --- |
| 可逆保证无主元 LU | 首主元为零的 $2\times2$ 反例可逆，换行后可分解 | 检查零主元、行列式和 $PA=LU$ |
| SPD 的普通 LU 因子互为转置 | 对 $H=[(4,2),(2,2)]$，$U=DL_0^T$，与 $L_0^T$ 的最大逐项差为 3 | 手算因子和 NumPy Cholesky 相对照 |
| 乘数有界即可避免消元增长 | 四阶测试矩阵乘数最大绝对值为 1，增长因子为 8 | 显式验证因子和逐步消元中的最大值 |
| Cholesky 可用于任意实方阵 | 对称不定例应失败。非对称例的下三角补成 SPD 后，库调用可能返回因子 | 检验 $LL^T$ 对完整矩阵的残差 |

In [2]:
a = np.array([[0., 1.], [1., 1.]])
p = np.array([[0., 1.], [1., 0.]])
close("4.1-01 非奇异但首主元为零", [np.linalg.det(a), a[0, 0]], [-1., 0.])
close("4.1-02 换行后的 LU", p @ a, np.array([[1., 1.], [0., 1.]]))

h = np.array([[4., 2.], [2., 2.]])
l0 = np.array([[1., 0.], [.5, 1.]])
d = np.diag([4., 1.])
u = d @ l0.T
close("4.1-03 普通 LU 和 LDL^T", l0 @ u, h)
close("4.1-04 U 与 L0^T 的最大逐项差", np.max(np.abs(u - l0.T)), 3.)
close("4.1-05 正对角 Cholesky 因子", np.linalg.cholesky(h), l0 @ np.sqrt(d))

# 部分选主元遇并列时取最先出现的行。这个矩阵执行时每列都保留对角行。
w = np.eye(4) - np.tril(np.ones((4, 4)), -1)
w[:, -1] = 1.
upper = w.copy()
max_intermediate = np.max(np.abs(w))
multipliers = []
for k in range(3):
    assert np.argmax(np.abs(upper[k:, k])) == 0
    factors = upper[k + 1:, k] / upper[k, k]
    multipliers.extend(np.abs(factors))
    upper[k + 1:, k:] -= factors[:, None] * upper[k, k:]
    max_intermediate = max(max_intermediate, np.max(np.abs(upper)))
close("4.1-06 乘数界和元素增长可以同时出现", [max(multipliers), max_intermediate], [1., 8.])

try:
    np.linalg.cholesky(np.array([[1., 2.], [2., 1.]]))
except np.linalg.LinAlgError:
    check("4.1-07 对称不定输入按预期失败", True)
else:
    raise AssertionError("对称不定矩阵不应成功分解")

# A1/A2 教学实现有对称性检查。此处测试直接调用库时的输入边界。
nonsymmetric = np.array([[2., 3.], [0., 2.]])
chol = np.linalg.cholesky(nonsymmetric)
close("4.1-08 库读取下三角时，完整矩阵分解残差", np.linalg.norm(chol @ chol.T - nonsymmetric, 'fro'), 3.)

通过 | 4.1-01 非奇异但首主元为零
       实测 [np.float64(-1.0), np.float64(0.0)] | 参考 [-1.0, 0.0]
通过 | 4.1-02 换行后的 LU
       实测 [[1. 1.]
 [0. 1.]] | 参考 [[1. 1.]
 [0. 1.]]
通过 | 4.1-03 普通 LU 和 LDL^T
       实测 [[4. 2.]
 [2. 2.]] | 参考 [[4. 2.]
 [2. 2.]]
通过 | 4.1-04 U 与 L0^T 的最大逐项差
       实测 3.0 | 参考 3.0
通过 | 4.1-05 正对角 Cholesky 因子
       实测 [[2. 0.]
 [1. 1.]] | 参考 [[2. 0.]
 [1. 1.]]
通过 | 4.1-06 乘数界和元素增长可以同时出现
       实测 [np.float64(1.0), np.float64(8.0)] | 参考 [1.0, 8.0]
通过 | 4.1-07 对称不定输入按预期失败
通过 | 4.1-08 库读取下三角时，完整矩阵分解残差
       实测 3.0 | 参考 3.0


## 4.2 迭代法 · CG 与预条件

| 被审结论 | 独立预期 | 判断标准 |
| --- | --- | --- |
| 小残差能单独保证小解误差 | SPD 对角系统的相对残差 $10^{-10}$，相对解误差 $1/\sqrt2$ | 分别对原右端、扰动后右端核验 |
| A1 的三对角一轮收敛能描述一般速度 | 正弦右端激发一个模态。改用 $b=T\mathbf1$，八阶例需要四步 | 固定矩阵、初值与停止阈值 $10^{-10}$ |
| SPD 的 $A,M$ 使 $M^{-1}A$ 欧氏对称，且与对称变换有相同二范数条件数 | 共享正特征值 $0.5,1.5$，但 $\kappa_2$ 一个约 36、一个为 3 | 同时检查 $MB=B^TM$ 与奇异值定义的条件数 |

原稿计时按[报告](C2组-批判报告.md)列出的保存输出审核。本页进行迭代和适用条件核验。

In [3]:
a = np.diag([1., 1e-10])
b = np.array([1., 1e-10])
x_star = np.ones(2)
x_candidate = np.array([1., 2.])
b_changed = np.array([1., 2e-10])
rho = np.linalg.norm(b - a @ x_candidate) / np.linalg.norm(b)
err = np.linalg.norm(x_star - x_candidate) / np.linalg.norm(x_star)
close("4.2-01 原右端相对残差", rho, 1e-10, atol=1e-21, rtol=1e-12)
close("4.2-02 原系统前向误差", err, 1 / np.sqrt(2))
close("4.2-03 扰动后右端残差", np.linalg.norm(b_changed - a @ x_candidate), 0., atol=1e-21)


def cg_steps(a, b, rtol=1e-10):
    """独立的小规模 CG，x0=0，返回达到真实残差阈值的步数。"""
    x = np.zeros_like(b)
    r = b.copy()
    p = r.copy()
    rs = r @ r
    threshold = rtol * np.linalg.norm(b)
    if np.sqrt(rs) <= threshold:
        return 0
    for step in range(1, len(b) + 1):
        ap = a @ p
        alpha = rs / (p @ ap)
        x += alpha * p
        r -= alpha * ap
        if np.linalg.norm(b - a @ x) <= threshold:
            return step
        next_rs = r @ r
        p = r + (next_rs / rs) * p
        rs = next_rs
    raise AssertionError("小型 SPD 对照未在预期步数范围内达到容差")


n = 8
t = 2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)
v = np.sin(np.pi * np.arange(1, n + 1) / (n + 1))
lam = 2 - 2 * np.cos(np.pi / (n + 1))
close("4.2-04 正弦制造解的特征残差", np.linalg.norm(t @ v - lam * v), 0.)
close("4.2-05 两种右端的实际步数", [cg_steps(t, t @ v), cg_steps(t, t @ np.ones(n))], [1, 4], atol=0.)

a = np.array([[100., 5.], [5., 1.]])
m = np.diag([100., 1.])
w = np.diag([.1, 1.])
left = np.linalg.solve(m, a)
symmetric = w @ a @ w
close("4.2-06 左预条件矩阵的欧氏反对称差", np.max(np.abs(left - left.T)), 4.95)
close("4.2-07 M 内积自伴性", np.linalg.norm(m @ left - left.T @ m, 'fro'), 0.)
close("4.2-08 共享谱", np.sort(np.linalg.eigvals(left)), np.array([.5, 1.5]))
trace = 27.0025
large_gram_eig = (trace + np.sqrt(trace**2 - 4 * .75**2)) / 2
close("4.2-09 二范数条件数不同", np.linalg.cond(left), large_gram_eig / .75)
close("4.2-10 PCG 对称变换的条件数", np.linalg.cond(symmetric), 3.)

通过 | 4.2-01 原右端相对残差
       实测 1e-10 | 参考 1e-10
通过 | 4.2-02 原系统前向误差
       实测 0.7071067811865475 | 参考 0.7071067811865475
通过 | 4.2-03 扰动后右端残差
       实测 0.0 | 参考 0.0
通过 | 4.2-04 正弦制造解的特征残差
       实测 3.3716170447940345e-16 | 参考 0.0
通过 | 4.2-05 两种右端的实际步数
       实测 [1, 4] | 参考 [1, 4]
通过 | 4.2-06 左预条件矩阵的欧氏反对称差
       实测 4.95 | 参考 4.95
通过 | 4.2-07 M 内积自伴性
       实测 0.0 | 参考 0.0
通过 | 4.2-08 共享谱
       实测 [0.5 1.5] | 参考 [0.5 1.5]
通过 | 4.2-09 二范数条件数不同
       实测 35.975536666691035 | 参考 35.975536666691085
通过 | 4.2-10 PCG 对称变换的条件数
       实测 2.999999999999999 | 参考 3.0


## 4.3 特征值、SVD 与扰动界

| 被审结论 | 独立预期 | 判断标准 |
| --- | --- | --- |
| 非对称矩阵的奇异值可用特征值模替代 | 固定 Jordan 块的特征值均为 1，奇异值为 $(\sqrt5\pm1)/2$ | 分别计算两套谱，与解析值比较 |
| 固定非正规矩阵的特征值与奇异值有相同的扰动行为 | $\lVert E\rVert_2=10^{-8}$ 时两特征值移动 $10^{-4}$。奇异值移动不超过 $\lVert E\rVert_2$ | 检查特征值移动与 Weyl 界 |
| A2 式 (4.15) 可作为有限系数扰动的严格界 | 对角例的实测为 $1/3$，旧式右端 $1/4$，带分母的上界取等 | 固定右端，独立比较旧式和修正式 |
| 高重构能量可推出高预测能力 | 秩 1 保留 $100/101$ 能量，但丢失标签方向 | 训练集拟合两套回归，分别在另设测试点上评价 |

In [4]:
j = np.array([[1., 1.], [0., 1.]])
eps = 1e-8
e = np.array([[0., 0.], [eps, 0.]])
s = np.linalg.svd(j, compute_uv=False)
close("4.3-01 Jordan 特征值", np.linalg.eigvals(j), np.ones(2))
close("4.3-02 Jordan 奇异值", s, (np.sqrt(5) + np.array([1., -1.])) / 2)
eig_move = np.max(np.abs(np.linalg.eigvals(j + e) - 1))
sv_move = np.max(np.abs(np.linalg.svd(j + e, compute_uv=False) - s))
close("4.3-03 特征值移动为 sqrt(eps)", eig_move, np.sqrt(eps))
check("4.3-04 奇异值移动满足 Weyl", sv_move <= eps + ATOL)
print(f"       eig 移动 {eig_move:.9g} | sv 移动 {sv_move:.9g} | ||E||₂ {eps:.9g}")
close("4.3-05 Weyl 上界可取等", np.linalg.svd(np.eye(2) * (1 + eps), compute_uv=False) - 1,
      np.full(2, eps))

a = np.diag([2., 1.])
delta = np.diag([0., -.25])
b = np.array([0., 1.])
x = np.linalg.solve(a, b)
x_perturbed = np.linalg.solve(a + delta, b)
error = np.linalg.norm(x_perturbed - x) / np.linalg.norm(x)
eta = np.linalg.cond(a) * np.linalg.norm(delta, 2) / np.linalg.norm(a, 2)
close("4.3-06 旧右端及真误差", [eta, error], [.25, 1 / 3])
check("4.3-07 实测超过旧界", error > eta)
close("4.3-08 带分母的严格界取等", error, eta / (1 - eta))

# 列是样本。两个特征均为零均值，训练集主方向是 e1。
x_train = np.array([[10., 10., -10., -10.], [1., -1., 1., -1.]])
x_test = np.array([[20., 20., -20., -20.], [-1., 1., -1., 1.]])
y_train, y_test = x_train[1], x_test[1]
directions, values, _ = np.linalg.svd(x_train, full_matrices=False)
project = directions[:, :1].T
w_full = np.linalg.lstsq(x_train.T, y_train, rcond=None)[0]
w_rank1 = np.linalg.lstsq((project @ x_train).T, y_train, rcond=None)[0]
mse_full = np.mean((x_test.T @ w_full - y_test)**2)
mse_rank1 = np.mean(((project @ x_test).T @ w_rank1 - y_test)**2)
close("4.3-09 秩1的能量比例", values[0]**2 / (values @ values), 100 / 101)
close("4.3-10 全特征与秩1分支的测试 MSE", [mse_full, mse_rank1], [0., 1.])

通过 | 4.3-01 Jordan 特征值
       实测 [1. 1.] | 参考 [1. 1.]
通过 | 4.3-02 Jordan 奇异值
       实测 [1.61803399 0.61803399] | 参考 [1.61803399 0.61803399]
通过 | 4.3-03 特征值移动为 sqrt(eps)
       实测 9.999999999998899e-05 | 参考 0.0001
通过 | 4.3-04 奇异值移动满足 Weyl
       eig 移动 0.0001 | sv 移动 7.23606797e-09 | ||E||₂ 1e-08
通过 | 4.3-05 Weyl 上界可取等
       实测 [9.99999994e-09 9.99999994e-09] | 参考 [1.e-08 1.e-08]
通过 | 4.3-06 旧右端及真误差
       实测 [np.float64(0.25), np.float64(0.33333333333333326)] | 参考 [0.25, 0.3333333333333333]
通过 | 4.3-07 实测超过旧界
通过 | 4.3-08 带分母的严格界取等
       实测 0.33333333333333326 | 参考 0.3333333333333333
通过 | 4.3-09 秩1的能量比例
       实测 0.9900990099009901 | 参考 0.9900990099009901
通过 | 4.3-10 全特征与秩1分支的测试 MSE
       实测 [np.float64(1.232595164407831e-32), np.float64(1.0)] | 参考 [0.0, 1.0]


## 4.4 可微求解器、PCA 与 LoRA

| 被审结论 | 独立预期 | 判断标准 |
| --- | --- | --- |
| 对称输入的梯度测试能检测转置写错 | 对称例无法区分 $A^{-T}$ 与 $A^{-1}$；非对称例的两个方向导数分别为 $0.5,-0.5$ | 用独立解析值及中心差分验证正确实现，并注入错误转置 |
| PCA 按样本内部减均值等价于特征中心化 | 两维四样本的正确中心化给出零特征均值；按样本中心化后为 $(-5,5)$ | 比较均值及中心化前后的主方向 |
| $V^T\tilde X$ 可作主成分得分或在所有版本中都无法相乘 | 完整形式无法相乘；薄形式形状可能碰巧匹配，数值仍错误 | 与 $U^T\tilde X=\Sigma V^T$ 比较 |
| LoRA 等价于截断 $W_0$，或只看 $X$ 的方差就能判定任务更新方向 | 冻结 $W_0$ 的秩一增量和截断基座方向不同。岭回归例的更新可全在小方差方向 | 比较预测映射、维度及独立的岭回归参考值 |

In [5]:
a = np.array([[2., 1.], [0., 1.]])
b = np.array([3., 1.])
x = np.linalg.solve(a, b)
z = np.linalg.solve(a.T, x)
z_wrong = np.linalg.solve(a, x)
db = np.array([1., 0.])
da = np.array([[0., 0.], [0., 1.]])


def loss(a, b):
    x = np.linalg.solve(a, b)
    return 0.5 * (x @ x)


h = 1e-5
fd_b = (loss(a, b + h * db) - loss(a, b - h * db)) / (2 * h)
fd_a = (loss(a + h * da, b) - loss(a - h * da, b)) / (2 * h)
close("4.4-01 非对称矩阵的伴随变量", z, np.array([.5, .5]))
close("4.4-02 两个伴随方向导数", [z @ db, -z @ da @ x], [.5, -.5])
close("4.4-03 两个方向的中心差分", [fd_b, fd_a], [.5, -.5], atol=1e-8)
close("4.4-04 注入错误转置时的两个导数", [z_wrong @ db, -z_wrong @ da @ x], [0., -1.])

通过 | 4.4-01 非对称矩阵的伴随变量
       实测 [0.5 0.5] | 参考 [0.5 0.5]
通过 | 4.4-02 两个伴随方向导数
       实测 [np.float64(0.5), np.float64(-0.5)] | 参考 [0.5, -0.5]
通过 | 4.4-03 两个方向的中心差分
       实测 [np.float64(0.5000000000032756), np.float64(-0.5000000001809113)] | 参考 [0.5, -0.5]
通过 | 4.4-04 注入错误转置时的两个导数
       实测 [np.float64(0.0), np.float64(-1.0)] | 参考 [0.0, -1.0]


In [6]:
x = np.array([[-2., 2., -2., 2.], [9., 9., 11., 11.]])
x_centered = x - x.mean(axis=1, keepdims=True)
x_col_centered = x - x.mean(axis=0, keepdims=True)
close("4.4-05 按特征中心化", x_centered.mean(axis=1), np.zeros(2))
close("4.4-06 按样本中心化后特征均值不为零", x_col_centered.mean(axis=1), [-5., 5.])
close("4.4-07 按样本中心化只让样本内和为零", x_col_centered.sum(axis=0), np.zeros(4))
close("4.4-08 错误中心化并不会把数据抹成零", np.max(np.abs(x_col_centered)), 6.5)

u, s, vt = np.linalg.svd(x_centered, full_matrices=False)
u_uncentered, _, _ = np.linalg.svd(x, full_matrices=False)
close("4.4-09 中心化前后主方向正交", abs(u[:, 0] @ u_uncentered[:, 0]), 0.)
scores = u.T @ x_centered
close("4.4-10 正确的得分公式", scores, s[:, None] * vt)
wrong_thin = vt.T @ x_centered
close("4.4-11 形状碰巧匹配的替代错式输出形状", wrong_thin.shape, (4, 4), atol=0.)
_, _, vt_full = np.linalg.svd(x_centered, full_matrices=True)
try:
    vt_full @ x_centered
except ValueError:
    check("4.4-12 完整 V^T 乘 Xc 维度不相容", True)
else:
    raise AssertionError("完整 V^T @ Xc 应拒绝该形状")
try:
    vt @ x_centered
except ValueError:
    check("4.4-13 NumPy 标准薄 V^T 乘 Xc 也不相容", True)
else:
    raise AssertionError("标准薄 V^T @ Xc 应拒绝该形状")

通过 | 4.4-05 按特征中心化
       实测 [0. 0.] | 参考 [0. 0.]
通过 | 4.4-06 按样本中心化后特征均值不为零
       实测 [-5.  5.] | 参考 [-5.0, 5.0]
通过 | 4.4-07 按样本中心化只让样本内和为零
       实测 [0. 0. 0. 0.] | 参考 [0. 0. 0. 0.]
通过 | 4.4-08 错误中心化并不会把数据抹成零
       实测 6.5 | 参考 6.5
通过 | 4.4-09 中心化前后主方向正交
       实测 0.0 | 参考 0.0
通过 | 4.4-10 正确的得分公式
       实测 [[-2.  2. -2.  2.]
 [-1. -1.  1.  1.]] | 参考 [[-2.  2. -2.  2.]
 [-1. -1.  1.  1.]]
通过 | 4.4-11 形状碰巧匹配的替代错式输出形状
       实测 (4, 4) | 参考 (4, 4)
通过 | 4.4-12 完整 V^T 乘 Xc 维度不相容
通过 | 4.4-13 NumPy 标准薄 V^T 乘 Xc 也不相容


4.4-11 测试的错式是将 NumPy 薄 SVD 返回的 `Vt` 再转置，即 $V_r\tilde X$。B1 的公开 AI 对话把形状 $n\times d$ 称为薄 $V^T$。按标准约定，薄 $V_r^T$ 实为 $d\times n$（此例 $2\times4$）。两种写法都需要对照形状和语义。方阵形状即便合法，$V^T\tilde X$ 一般也不是得分。

In [7]:
# 4 个样本也可用 4 个特征表示。重用这 2 个特征，再补 2 个恒零特征。
# 此时 V^T X 维度合法，单凭形状看不出错。比较每行得分能量，规避符号歧义。
x_square = np.vstack([x_centered, np.zeros((2, 4))])
u_square, _, vt_square = np.linalg.svd(x_square, full_matrices=True)
correct = u_square.T @ x_square
wrong = vt_square @ x_square
check("4.4-14 方阵时 V^T X 可计算但给不出正确首主成分能量",
      abs(np.linalg.norm(correct[0]) - np.linalg.norm(wrong[0])) > .1)
print(f"       首行范数 正确={np.linalg.norm(correct[0]):.9g} | 错式={np.linalg.norm(wrong[0]):.9g}")

通过 | 4.4-14 方阵时 V^T X 可计算但给不出正确首主成分能量
       首行范数 正确=4 | 错式=2.23606798


### 对照两份 A 稿的 PCA 测例

重建 A1 实验单元索引 36 的 $6\times4$ 故障注入数据，以及 A2 实验单元索引 23 的 $5\times240$ 主数据。A1 的参考角度约 $1.30^\circ$ 来自 B1 的同一小例记录，只用于重现该结果。A2 由等间距三角函数正交性知均值方向为 $e_2$，中心化和未中心化的首方向分别为 $e_1,e_2$。因此 B1 汇总表不能把 A2 的“未中心化方向与均值重合”写成“中心化前后首方向重合”。

In [8]:
def first_direction_rows(samples, center):
    values = samples - samples.mean(axis=0, keepdims=True) if center else samples
    return np.linalg.svd(values, full_matrices=False)[2][0]


a1_demo = np.array([[1., 1., 1., 1.], [2., 3., 2., 3.], [3., 2., 4., 2.],
                    [4., 4., 3., 5.], [5., 6., 5., 6.], [6., 5., 7., 5.]])
a1_c = first_direction_rows(a1_demo, True)
a1_unc = first_direction_rows(a1_demo, False)
a1_angle = np.degrees(np.arccos(np.clip(abs(a1_c @ a1_unc), 0., 1.)))
close("4.4-20 A1 故障注入前后方向很接近", a1_angle, 1.30, atol=.01)

n = 240
phase = 2 * np.pi * np.arange(n) / n
a2_demo = np.column_stack([3 * np.cos(phase), 20 + np.sin(phase),
                           .2 * np.sin(3 * phase), .05 * np.cos(2 * phase),
                           .01 * np.sin(4 * phase)])
a2_c = first_direction_rows(a2_demo, True)
a2_unc = first_direction_rows(a2_demo, False)
close("4.4-21 A2 的中心化与未中心化首方向",
      [abs(a2_c[0]), abs(a2_unc[1]), abs(a2_c @ a2_unc)], [1., 1., 0.])

通过 | 4.4-20 A1 故障注入前后方向很接近
       实测 1.2955015277705242 | 参考 1.3
通过 | 4.4-21 A2 的中心化与未中心化首方向
       实测 [np.float64(1.0), np.float64(0.9999999999999999), np.float64(0.0)] | 参考 [1.0, 1.0, 0.0]


In [9]:
w0 = np.diag([3., 1.])
b_factor = np.array([[0.], [1.]])
a_factor = b_factor.T
update = b_factor @ a_factor
adapted = w0 + update
u_lora, s_lora, vt_lora = np.linalg.svd(w0)
truncated = (u_lora[:, :1] * s_lora[:1]) @ vt_lora[:1]
probe = np.array([0., 1.])
close("4.4-15 矩阵增量秩", np.linalg.matrix_rank(update), 1, atol=0.)
close("4.4-16 LoRA 与原权重截断在 e2 上的输出范数",
      [np.linalg.norm(adapted @ probe), np.linalg.norm(truncated @ probe)], [2., 0.])
close("4.4-17 64×64、r=4 适配器参数量", 64 * 4 + 4 * 64, 512, atol=0.)

features = np.array([[1., -1., 1., -1.], [.01, .01, -.01, -.01]])
target_residual = np.array([1., 1., -1., -1.])
cov = features @ features.T / features.shape[1]
rhs = features @ target_residual / features.shape[1]
gamma = 1e-4
delta_w = np.linalg.solve(cov + gamma * np.eye(2), rhs)
close("4.4-18 岭回归更新落在小方差方向", delta_w, np.array([0., 50.]))
close("4.4-19 岭回归逆谱系数", 1 / (np.diag(cov) + gamma), [1 / 1.0001, 5000.])

通过 | 4.4-15 矩阵增量秩
       实测 1 | 参考 1
通过 | 4.4-16 LoRA 与原权重截断在 e2 上的输出范数
       实测 [np.float64(2.0), np.float64(0.0)] | 参考 [2.0, 0.0]
通过 | 4.4-17 64×64、r=4 适配器参数量
       实测 512 | 参考 512
通过 | 4.4-18 岭回归更新落在小方差方向
       实测 [ 0. 50.] | 参考 [ 0. 50.]
通过 | 4.4-19 岭回归逆谱系数
       实测 [9.9990001e-01 5.0000000e+03] | 参考 [0.9999000099990001, 5000.0]


## 反例、适用边界与复核结论

4.1 支持 A1 的条件说明及 B2 的 Cholesky 改法。4.2 支持 B1 对正弦测例的判断和 B2 对 PCG 相似变换的改法。4.3 支持给系数扰动上界加分母，低秩重构只按指定误差指标评价。4.4 支持用非对称矩阵作梯度回归测试，并区分特征中心化、监督目标与低秩参数化。逐项修订和待复测状态见[批判报告](C2组-批判报告.md)。

输出中的“通过”表示实测符合本次预先写出的参考结果，包含按预期失败的负例，不等于被审教材全章验收通过。B1/B2 的运行与版本状态以[报告](C2组-批判报告.md)为准。

In [10]:
print(f"完成 {len(passed)} 项断言，全部通过。")

完成 49 项断言，全部通过。
